In [1]:
from google.colab import userdata
import os
from huggingface_hub import login
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
login(userdata.get("HF_API_KEY"))

In [2]:
!kaggle competitions download -c super-ai-engineer-ss-6-thai-language-image-captioning

100% 1.75G/1.75G [01:40<00:00, 18.7MB/s]



In [3]:
!unzip "/content/super-ai-engineer-ss-6-thai-language-image-captioning.zip"
!rm "/content/super-ai-engineer-ss-6-thai-language-image-captioning.zip"

Streaming output truncated to the last 5000 lines.
  inflating: train/train/travel/15347.jpg  
  inflating: train/train/travel/15348.jpg  
  inflating: train/train/travel/15349.jpg  
  inflating: train/train/travel/15350.jpg  
  inflating: train/train/travel/15351.jpg  
  inflating: train/train/travel/15352.jpg  
  inflating: train/train/travel/15353.jpg  
  inflating: train/train/travel/15354.jpg  
  inflating: train/train/travel/15355.jpg  
  inflating: train/train/travel/15356.jpg  
  inflating: train/train/travel/15357.jpg  
  inflating: train/train/travel/15358.jpg  
  inflating: train/train/travel/15359.jpg  
  inflating: train/train/travel/15360.jpg  
  inflating: train/train/travel/15361.jpg  
  inflating: train/train/travel/15362.jpg  
  inflating: train/train/travel/15363.jpg  
  inflating: train/train/travel/15364.jpg  
  inflating: train/train/travel/15365.jpg  
  inflating: train/train/travel/15366.jpg  
  inflating: train/train/travel/15367.jpg  
  inflating: train/train/

In [4]:
!pip install unsloth peft accelerate bitsandbytes trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 165.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [5]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
from unsloth import FastVisionModel
from transformers import AutoTokenizer
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
    # use_gradient_checkpointing = "unsloth",
    # fast_inference = True
)
model.load_adapter("KiruruP/Qwen3-VL-8B-ipu10k-thaicaption-lora")
# tokenizer = AutoTokenizer.from_pretrained("KiruruP/Qwen3-VL-8B-ipu10k-thaicaption-lora")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/103M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/720 [00:00<?, ?it/s]

In [14]:
from tqdm import tqdm
from PIL import Image

FastVisionModel.for_inference(model)

instruction = "จงบรรยายภาพนี้เป็นภาษาไทยแบบสั้น กระชับ และเป็นธรรมชาติ"
test_cap = []

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True, tokenize = False)

def generateCaption(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")
    output = model.generate(**inputs, max_new_tokens = 60,
                      use_cache = True, temperature = 0.5, min_p = 0.1)
    caption = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    test_cap.append(caption)

In [15]:
folder_path = '/content/test/test'

image_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
            image_paths.append(os.path.join(root, file))

print(len(image_paths))

2000


In [16]:
with torch.inference_mode():
  for img_path in tqdm(image_paths):
      generateCaption(img_path)

100%|██████████| 2000/2000 [3:50:43<00:00,  6.92s/it]


In [17]:
image_id = [k.split('/')[-1] for k in image_paths]

In [18]:
image_id_real = [k.split('.')[0] for k in image_id]

In [20]:
import pandas as pd

data = {'image_id': image_id_real, 'cc': test_cap}

# Create a DataFrame
submission = pd.DataFrame(data)

In [21]:
submission['cc'] = submission['cc'].apply(lambda x: x.split('\n')[-1])
submission['cc'] = submission['cc'].apply(lambda x: x.replace('"',''))

In [22]:
sample = pd.read_csv('/content/sample_submission.csv', dtype={'image_id': str})

In [23]:
sampleid = sample['image_id']

In [24]:
id_list = sampleid.to_list()

In [25]:
df_final = submission.set_index('image_id').loc[id_list].reset_index()

In [26]:
df_final = df_final.rename(columns={'cc': 'caption'})

In [27]:
df_final.to_csv('image_captioning_submission.csv', index=False)

In [28]:
from google.colab import files
files.download("image_captioning_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>